# Day 1 — Setup & scope lock (GPU)

Run this on a CUDA GPU (Colab T4/A100, RunPod, etc.). It covers the parts of `docs/tasks/day-1.md`
that need the model:

- **S1.1** — load `unsloth/Qwen3-1.7B-unsloth-bnb-4bit`, run one inference call in **non-thinking mode**, confirm no chain-of-thought leaks.
- **S1.3** — serialize one training-format example and eyeball prompt-vs-completion boundaries.

The tag syntax + prompt come from the repo (`src/common/tags.py`, `src/common/prompts.py`) so the
notebook and training code can't drift. Those tags tests already pass locally (S1.2).

> After running: paste the serialized example into `docs/tasks/artifacts/day1-serialized-example.txt`
> and fill the decisions block in `docs/tasks/day-1.md`."

## 0. Environment + repo

In [ ]:
# On Colab: clone the repo so `src/` is importable. Locally, skip the clone.
import os, sys

REPO_HOST = "github.com/f15cubing/slm-deid.git"
REPO_DIR = "slm-deid"

def _have_src():
    return os.path.isdir("src/common")

if not _have_src():
    # 1) Try a plain clone — works if the repo is PUBLIC.
    os.system(f"git clone https://{REPO_HOST} {REPO_DIR}")
    # 2) Private repo? A plain clone fails (asks for a username). Fall back to a token.
    if not os.path.isdir(REPO_DIR):
        from getpass import getpass
        token = getpass("Repo is private — paste a GitHub token (repo scope), or blank to abort: ").strip()
        if token:
            os.system(f"git clone https://{token}@{REPO_HOST} {REPO_DIR}")
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)

assert _have_src(), (
    "Repo not available. Either make github.com/f15cubing/slm-deid PUBLIC, "
    "or re-run and paste a GitHub token with 'repo' scope."
)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

# Unsloth pulls a compatible torch/transformers/trl/peft/bitsandbytes stack.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || pip install -q unsloth

## 1. Load base model (S1.1)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"  # Day-1 decision (see docs/tasks/day-1.md)
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model)  # 2x faster inference
print("loaded:", MODEL_NAME)

## 2. One inference call, non-thinking mode (S1.1)

`enable_thinking=False` + an empty `<think></think>` template. We assert no thinking content leaks
into the output.

In [ ]:
import torch
from src.common import prompts

def tag(passage: str, max_new_tokens: int = 256) -> str:
    messages = prompts.build_messages(passage)
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,          # non-thinking mode
        return_tensors="pt",
        return_dict=True,               # input_ids + attention_mask (silences the mask warning)
    ).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,                # deterministic for a clean Day-1 read
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    completion = tokenizer.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True)
    return completion.strip()

demo_passage, note = prompts.SHOWCASE[0]
out = tag(demo_passage)
print("INPUT   :", demo_passage)
print("EXPECTED:", note)
print("OUTPUT  :", out)

assert "<think>" not in out and "</think>" not in out, "thinking content leaked into output!"
print("\n[OK] S1.1 — model responds, no thinking leak.")

## 3. Prompted-base behavior on the ambiguous showcase (early signal)

Not scored yet — just a first look at where the *prompted* base wobbles (over-tagging "the Newton
method", missing first-name-only, etc.). This is exactly the gap the fine-tune must close (SPOV 7).

In [ ]:
from src.common import tags

for passage, note in prompts.SHOWCASE:
    out = tag(passage)
    integrity_ok = tags.unwrap(out) == passage
    well_formed = tags.is_well_formed(out)
    flag = "" if (integrity_ok and well_formed) else "  <-- INTEGRITY/FORMAT ISSUE"
    print(f"- {note}{flag}")
    print(f"    in : {passage}")
    print(f"    out: {out}\n")

## 4. Serialize one training example + tag-syntax tokenizer test (S1.3)

Confirms (a) the chat template renders prompt + tagged completion with a clear boundary, and (b) the
`⟨NAME⟩` markers survive tokenization intact (the Day-1 tag-syntax decision). If the angle brackets
get shredded into many pieces, switch `src/common/tags.py` to the `@@…##` fallback and re-run.

In [ ]:
import pathlib

# A representative gold example: person "Chelsea" tagged, place "Chelsea" left alone.
ex_input = "Chelsea helped me revise, but last year I also visited Chelsea in London."
ex_target = f"{tags.wrap('Chelsea')} helped me revise, but last year I also visited Chelsea in London."
assert tags.unwrap(ex_target) == ex_input  # integrity invariant

serialized = prompts.serialize(
    tokenizer,
    prompts.build_training_messages(ex_input, ex_target),
    add_generation_prompt=False,   # full example incl. assistant turn
)
print(serialized)

# Tag-syntax tokenizer test: how many tokens do the markers cost, and do they round-trip?
for marker_name, marker in [("OPEN", tags.NAME_OPEN), ("CLOSE", tags.NAME_CLOSE)]:
    ids = tokenizer.encode(marker, add_special_tokens=False)
    roundtrip = tokenizer.decode(ids)
    print(f"{marker_name} {marker!r}: {len(ids)} tokens, roundtrip={'OK' if roundtrip == marker else 'BROKEN ' + repr(roundtrip)}")

# Save the artifact for S1.3.
art = pathlib.Path("docs/tasks/artifacts/day1-serialized-example.txt")
art.parent.mkdir(parents=True, exist_ok=True)
art.write_text(serialized, encoding="utf-8")
print("\n[OK] S1.3 — wrote", art)